# 02 — Transfermarkt Data Cleaning

**Goal:** Extract and clean player valuations from the Transfermarkt Kaggle dataset for our 7 target leagues during the 2023-24 season.

**Input:** `data/raw/transfermarkt/players.csv`, `data/raw/transfermarkt/player_valuations.csv`  
**Output:** `data/cleaned/transfermarkt_cleaned.csv`

## 1. Imports and Config

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR     = Path('../data/raw/transfermarkt')
CLEANED_DIR = Path('../data/cleaned')
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Transfermarkt competition IDs mapped to our league labels
TARGET_COMPS = {
    'NL1':  'Eredivisie',
    'PO1':  'Primeira Liga',
    'BE1':  'Belgian Pro League',
    'L1':   'Bundesliga',
    'FR1':  'Ligue 1',
    'SC1':  'Scottish Premiership',
    'BRA1': 'Brazil Serie A',
}

# Season window for valuation snapshot
SEASON_START = '2023-06-01'
SEASON_END   = '2024-06-30'

## 2. Load Raw Files

In [ ]:
players    = pd.read_csv(RAW_DIR / 'players.csv')
valuations = pd.read_csv(RAW_DIR / 'player_valuations.csv')

print(f'players.csv:           {players.shape}')
print(f'player_valuations.csv: {valuations.shape}')

## 3. Filter to Target Leagues and Select Columns

In [ ]:
tm = players[players['current_club_domestic_competition_id'].isin(TARGET_COMPS)].copy()
tm['league'] = tm['current_club_domestic_competition_id'].map(TARGET_COMPS)

tm = tm[[
    'player_id', 'name', 'current_club_name', 'league',
    'position', 'sub_position', 'date_of_birth',
    'country_of_citizenship', 'market_value_in_eur', 'highest_market_value_in_eur'
]].rename(columns={
    'name':                    'player_name',
    'current_club_name':       'club',
    'country_of_citizenship':  'nationality',
    'date_of_birth':           'dob',
})

print('Players in target leagues:', len(tm))
print(tm['league'].value_counts())

## 4. Get Season Valuation Snapshot

Use the most recent valuation recorded during the 2023-24 season window. Falls back to `market_value_in_eur` from `players.csv` if no in-season record exists.

In [ ]:
valuations['date'] = pd.to_datetime(valuations['date'])

season_vals = valuations[
    (valuations['date'] >= SEASON_START) &
    (valuations['date'] <= SEASON_END) &
    (valuations['player_club_domestic_competition_id'].isin(TARGET_COMPS))
]

latest_val = (
    season_vals
    .sort_values('date')
    .groupby('player_id')['market_value_in_eur']
    .last()
    .reset_index()
    .rename(columns={'market_value_in_eur': 'market_value_season'})
)

tm = tm.merge(latest_val, on='player_id', how='left')

# Use season value where available, fall back to current value from players.csv
tm['market_value_eur'] = tm['market_value_season'].fillna(tm['market_value_in_eur'])

print(f'Players with season valuation: {tm["market_value_season"].notna().sum()}')
print(f'Fallback to current value:     {tm["market_value_season"].isna().sum()}')
print(f'No valuation at all:           {tm["market_value_eur"].isna().sum()}')

## 5. Clean and Derive Columns

In [ ]:
# Age at start of 2023-24 season
tm['dob'] = pd.to_datetime(tm['dob'], errors='coerce')
season_ref = pd.Timestamp('2023-08-01')
tm['age'] = ((season_ref - tm['dob']).dt.days / 365.25).round(1)

# Market value in millions for readability
tm['market_value_m'] = (tm['market_value_eur'] / 1_000_000).round(3)

# Drop rows with no market value — can't score without it
before = len(tm)
tm = tm[tm['market_value_eur'].notna() & (tm['market_value_eur'] > 0)].copy()
print(f'Dropped {before - len(tm)} rows with no/zero market value')
print(f'Remaining: {len(tm)} players')

# Drop goalkeepers
before = len(tm)
tm = tm[tm['position'] != 'Goalkeeper'].copy()
print(f'Dropped {before - len(tm)} goalkeepers')
print(f'Remaining: {len(tm)} outfield players')

## 6. Position Standardisation

Transfermarkt uses broad position groups (Attack, Midfield, Defender). We keep `sub_position` for granularity and map to a simplified 3-category system for scoring.

In [ ]:
POSITION_MAP = {
    'Attack':   'Forward',
    'Midfield': 'Midfielder',
    'Defender': 'Defender',
}
tm['position_group'] = tm['position'].map(POSITION_MAP).fillna('Midfielder')

print('Position groups:')
print(tm['position_group'].value_counts())
print('\nSub-positions:')
print(tm['sub_position'].value_counts())

## 7. Final Columns and Save

In [ ]:
tm_clean = tm[[
    'player_id', 'player_name', 'club', 'league', 'nationality',
    'age', 'position', 'position_group', 'sub_position',
    'market_value_eur', 'market_value_m', 'highest_market_value_in_eur'
]].copy()

print('Final shape:', tm_clean.shape)
print('\nSample:')
tm_clean.head(5)

out = CLEANED_DIR / 'transfermarkt_cleaned.csv'
tm_clean.to_csv(out, index=False)
print(f'\nSaved to {out}')